# SynKit CRN: executable glycolysis from cached KEGG data

This notebook reproduces the paper's M00001 structural analysis without network access. It separates parameter-free structural results from the discrete firing certificate; neither is a kinetic simulation.

In [ ]:
from pathlib import Path
import sys

repo_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file() and (path / "synkit").is_dir()
)
sys.path.insert(0, str(repo_root))

from synkit.CRN import (
    PathwayRealizability, conserved_moieties, crnt_summary, find_siphons,
    siphon_persistence_details, write_sbml,
)
from synkit.CRN.Benchmark import (
    GLYCOLYSIS_FLUX, build_kegg_crn, check_glycolysis_flux,
)

## Load and summarize the normalized network

Currency metabolites are removed by the benchmark's documented default so the analysis describes the module's carbon skeleton rather than global cofactor recycling.

In [ ]:
crn = build_kegg_crn("M00001")
summary = crnt_summary(crn)
assert (summary.n_species, summary.n_reactions) == (14, 15)
assert summary.deficiency == 1
assert not summary.is_weakly_reversible
print(summary)

## Conserved moieties, siphons, and persistence

The persistence condition is sufficient, not necessary: failure identifies uncovered siphons but does not prove extinction under a kinetic model.

In [ ]:
moieties = conserved_moieties(crn)
siphons = find_siphons(crn)
persistence = siphon_persistence_details(crn)
assert len(moieties) == 3
assert len(siphons) == 3
assert not persistence.persistence_ok
print({"moieties": len(moieties), "siphons": len(siphons), "persistent_by_sufficient_test": persistence.persistence_ok})

## Find and independently replay the firing certificate

The augmented Petri net supplies each requested reaction with its prescribed firing count. Replaying the returned transition sequence checks that every step is enabled and that the exact target marking is reached.

In [ ]:
outcome = check_glycolysis_flux()
assert outcome["status"] == "realizable"
assert outcome["explored_states"] == 30
assert len(outcome["certificate"]) == 15

checker = PathwayRealizability().load_syncrn_and_flow(
    crn, flow=GLYCOLYSIS_FLUX,
    initial_marking={"alpha-D-Glucose": 1},
    species="label", reaction="label",
).build_petri_net_from_flow()
marking = checker.initial_marking
replay = [dict(marking)]
for transition in outcome["certificate"]:
    assert checker.petri.enabled(marking, transition)
    marking = checker.petri.fire(marking, transition)
    assert min(marking.values()) >= 0
    replay.append(dict(marking))
assert all(marking.get(place, 0) == value for place, value in checker.goal_exact.items())
print({"certificate_length": len(outcome["certificate"]), "replayed_markings": len(replay), "final_pyruvate": marking["Pyruvate"]})

## Export the same structural network as SBML

SynKit writes SBML Level 3 Version 2 Core. The structural model has no kinetic laws because none are represented by `SynCRN`.

In [ ]:
output = Path("glycolysis-m00001.xml")
write_sbml(crn, output, model_id="kegg_m00001")
assert output.exists() and "<sbml" in output.read_text(encoding="utf-8")
print(output.resolve())